# Figure 5 — expression (DESeq2 + GSEA)
RNA-seq (GEO GSE116862; recount3 SRP152979; 2 replicates per state) → DESeq2 (day 80 vs day 0) → preranked GSEA on
the DESeq2 Wald statistic (shared `../gsea.py`) for the 60-pathway panel (Hallmark + custom), the 14 published
cardiomyocyte sets, the 13 stemness sets and the 178 PanglaoDB cell types. Writes
`data/gsea_{hallmark,cardiac,stemness,celltype}.csv`, read by `psccm_analysis.ipynb`. Run first (needs pydeseq2 0.4.12 and
gseapy 1.3.1); ~2 min.

In [1]:
import json
import sys

import numpy as np
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

sys.path.insert(0, "..")                                # shared modules
import gsea                                             # preranked GSEA on the DESeq2 Wald statistic

DATA = "data"
META = "../../metadata"
GENE_SUMS = "/data/land/3d-inter-paper-data/psc-cm-gse116862/rnaseq/recount3/gene_sums.G026.gz"
GTF = f"{META}/hg38/hg38_gtf.csv"
HALLMARK_GMT = f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.gmt"
GSEA_COLUMNS = ["Term", "NES", "NOM p-val", "FDR q-val"]

panel = json.load(open(f"{META}/psc-cm/psc_cm_pathways.json"))["cardiomyocyte"]      # main panel (5b)
cardiac_sets = json.load(open(f"{META}/psc-cm/cardiomyocyte_published_sets.json"))
stem_sets = json.load(open(f"{META}/psc-cm/stemness_published_sets.json"))
celltype_sets = json.load(open(f"{META}/psc-cm/panglaodb_celltypes.json"))

## DESeq2

In [2]:
# recount3 run -> (label, condition, avg read length from ENA)
SAMPLES = {
    "SRR7506545": ("day00_Rep1", "day00", 198.674),
    "SRR7506546": ("day00_Rep2", "day00", 198.506),
    "SRR7506555": ("day80_Rep1", "day80", 198.856),
    "SRR7506556": ("day80_Rep2", "day80", 199.488),
}

coverage = pd.read_csv(GENE_SUMS, sep="\t", comment="#").set_index("gene_id")[list(SAMPLES)]   # base-pair coverage
counts = coverage.astype(float).copy()
for srr, (_, _, readlen) in SAMPLES.items():
    counts[srr] = np.rint(counts[srr] / readlen)          # coverage -> read counts
counts = counts.astype(int)
counts.columns = [SAMPLES[c][0] for c in counts.columns]

# ENSG (version-stripped) -> symbol, collapse duplicates
gtf = pd.read_csv(GTF, usecols=["gene_id", "gene_name"]).dropna().drop_duplicates("gene_id")
id2sym = dict(zip(gtf.gene_id, gtf.gene_name))
counts.index = [g.split(".")[0] for g in counts.index]
counts["symbol"] = [id2sym.get(g) for g in counts.index]
counts = counts.dropna(subset=["symbol"]).groupby("symbol").sum()
print(f"count matrix: {counts.shape[0]} genes x {counts.shape[1]} samples")

count matrix: 56211 genes x 4 samples


In [3]:
meta = pd.DataFrame({"condition": [c.rsplit("_", 1)[0] for c in counts.columns]}, index=counts.columns)
dds = DeseqDataSet(counts=counts.T, metadata=meta, design_factors="condition")
dds.deseq2()
stat = DeseqStats(dds, contrast=["condition", "day80", "day00"])
stat.summary()
res = stat.results_df.copy()
res.index.name = "gene_name"
print(f"{len(res)} genes; {(res.padj < 0.05).sum()} at padj < 0.05")

Fitting size factors...
... done in 0.01 seconds.



Fitting dispersions...


... done in 4.60 seconds.

Fitting dispersion trend curve...


... done in 0.66 seconds.

/local_home/land/anaconda3/envs/pydeseq2_env/lib/python3.11/site-packages/pydeseq2/dds.py:497: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...


... done in 4.47 seconds.

Fitting LFCs...


... done in 4.90 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: condition day80 vs day00
                   baseMean  log2FoldChange     lfcSE      stat        pvalue  \
symbol                                                                          
5S_rRNA            0.000000             NaN       NaN       NaN           NaN   
5_8S_rRNA          3.004682       -2.041814  2.271858 -0.898742  3.687901e-01   
7SK              117.802835        1.454943  0.513975  2.830765  4.643679e-03   
A1BG             205.238279        1.143903  0.392305  2.915849  3.547226e-03   
A1BG-AS1         284.621054        2.314353  0.352663  6.562506  5.291096e-11   
...                     ...             ...       ...       ...           ...   
ZZZ3            1668.315504       -0.275104  0.290492 -0.947026  3.436253e-01   
hsa-mir-1253       0.000000             NaN       NaN       NaN           NaN   
hsa-mir-423        0.288888        1.921785  5.346913  0.359420  7.192812e-01   
hsa-mir-8069-1     0.000000             NaN   

... done in 3.93 seconds.



## GSEA

In [4]:
rnk = gsea.deseq_rank(res)                              # ranked by the DESeq2 Wald statistic
gsea_results = {"hallmark": gsea.prerank(rnk, HALLMARK_GMT),
                "cardiac": gsea.prerank(rnk, cardiac_sets),
                "stemness": gsea.prerank(rnk, stem_sets),
                "celltype": gsea.prerank(rnk, celltype_sets)}
for name, result in gsea_results.items():
    result[GSEA_COLUMNS].to_csv(f"{DATA}/gsea_{name}.csv", index=False)
    print(f"{name}: {len(result)} sets, {(result['FDR q-val'] < 0.05).sum()} at FDR < 0.05")

2026-10-21 00:51:40,792 [WARNING] Duplicated values found in preranked stats: 29.96% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


2026-10-21 00:51:51,470 [WARNING] Duplicated values found in preranked stats: 29.96% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


2026-10-21 00:51:54,660 [WARNING] Duplicated values found in preranked stats: 29.96% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


2026-10-21 00:51:57,702 [WARNING] Duplicated values found in preranked stats: 29.96% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


hallmark: 60 sets, 26 at FDR < 0.05
cardiac: 14 sets, 14 at FDR < 0.05
stemness: 13 sets, 7 at FDR < 0.05
celltype: 178 sets, 95 at FDR < 0.05


In [5]:
cardiac_gsea = gsea_results["cardiac"]
cardiac_gsea[cardiac_gsea.Term.isin(panel)][GSEA_COLUMNS].round(4)    # main-panel sets (5b)

,Term,NES,NOM p-val,FDR q-val
0,Cardiomyocytes (PanglaoDB),3.0423,0.0,0.0
2,Sarcomere organization (GO:0045214),2.3854,0.0,0.0
6,Muscle Contraction (Reactome),2.3078,0.0,0.0
9,Hypertrophic cardiomyopathy (KEGG),2.1553,0.0,0.0
10,Adrenergic signaling in cardiomyocytes (KEGG),2.1409,0.0,0.0
